# Notebook 03: SHAP Explainability Analysis

SHAP (SHapley Additive exPlanations) analysis of the water conflict prediction model.  
Target journal: *Global Environmental Change*

**Key contribution**: Temporal SHAP decomposition tracking how driver importance shifts across geopolitical eras (Cold War, Post-Cold War, Post-2000).

## Cell 1: Setup

In [1]:
import pandas as pd
import numpy as np
import shap
import xgboost as xgb
from sklearn.impute import SimpleImputer
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import os
import warnings
warnings.filterwarnings('ignore')

# Ensure figures directory exists
FIGURES_DIR = '../figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

# Publication template
pub_template = dict(
    layout=dict(
        font=dict(family='Arial', size=12, color='black'),
        plot_bgcolor='white',
        paper_bgcolor='white',
        xaxis=dict(
            showgrid=False,
            linecolor='black',
            linewidth=1,
            mirror=True,
            ticks='outside'
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor='lightgray',
            linecolor='black',
            linewidth=1,
            mirror=True,
            ticks='outside'
        ),
        margin=dict(l=60, r=20, t=40, b=60),
    )
)

# Colour palette (colourblind-safe, Nature-style)
ERA_COLOURS = {
    'Cold War (1948-1990)':   '#1f77b4',
    'Post-Cold War (1991-1999)': '#ff7f0e',
    'Post-2000 (2000-2008)':  '#2ca02c',
}

CLASS_LABELS = {0: 'Conflict', 1: 'Neutral', 2: 'Mild Coop.', 3: 'Strong Coop.'}

print('Setup complete. shap version:', shap.__version__)
print('xgboost version:', xgb.__version__)

Setup complete. shap version: 0.51.0
xgboost version: 3.2.0


## Cell 2: Load data, prepare features, train XGBoost

In [2]:
# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_parquet('../data/processed/events_enriched.parquet')
print(f'Loaded {len(df):,} events x {df.shape[1]} columns')
print(f'Year range: {df["year"].min()} – {df["year"].max()}')
print(f'Target distribution:\n{df["target"].value_counts().sort_index()}')

# ── Feature set (45 retained features from ablation) ──────────────────────────
FEATURE_COLS_CANDIDATE = [
    # BCU spatial / hydrological
    'Area_km2_1', 'Area_km2_2',
    'Dams_Exist_1', 'Dams_Exist_2',
    'Dam_Plnd_1', 'Dam_Plnd_2',
    'EstDam24_1', 'EstDam24_2',
    'runoff_1', 'runoff_2',
    'withdrawal_1', 'withdrawal_2',
    'consumpt_1', 'consumpt_2',
    'HydroPolTe_1', 'HydroPolTe_2',
    'InstitVuln_1', 'InstitVuln_2',
    'NumberRipa_1', 'NumberRipa_2',
    'Wetlands_k_1', 'Wetlands_k_2',
    'PopDen2022_1', 'PopDen2022_2',
    # Event
    'NUMBER_OF_BASINS', 'NUMBER_OF_Countries', 'bilateral', 'Issue_Type1',
    'treaties_before_event',
    # Economic (WDI)
    'NY.GDP.PCAP.CD_wdi1', 'NY.GDP.PCAP.CD_wdi2',
    'MS.MIL.XPND.GD.ZS_wdi1', 'MS.MIL.XPND.GD.ZS_wdi2',
    'SP.POP.TOTL_wdi1', 'SP.POP.TOTL_wdi2',
    'ER.H2O.FWTL.ZS_wdi1', 'ER.H2O.FWTL.ZS_wdi2',
    'ER.H2O.INTR.PC_wdi1', 'ER.H2O.INTR.PC_wdi2',
    # Temporal
    'events_prior_5yr', 'cooperation_momentum', 'cold_war',
    'treaty_rate_5yr', 'event_escalation', 'year',
]

# Keep only features that actually exist in the dataset
FEATURE_COLS = [c for c in FEATURE_COLS_CANDIDATE if c in df.columns]
print(f'\nUsing {len(FEATURE_COLS)}/{len(FEATURE_COLS_CANDIDATE)} candidate features')
missing_feats = [c for c in FEATURE_COLS_CANDIDATE if c not in df.columns]
if missing_feats:
    print(f'Dropped (not in dataset): {missing_feats}')

# ── Temporal split ────────────────────────────────────────────────────────────
# year is nullable Int64; cast to plain numpy int before comparison
year_np    = df['year'].fillna(-1).to_numpy().astype(int)
train_mask = year_np < 1996
df_train = df[train_mask].reset_index(drop=True).copy()
df_all   = df.reset_index(drop=True).copy()  # reset so label index == position

X_train = df_train[FEATURE_COLS].astype(float)
y_train = df_train['target'].astype(int)

X_all = df_all[FEATURE_COLS].astype(float)
y_all = df_all['target'].astype(int)

print(f'\nTrain set (year < 1996): {len(X_train):,} events')
print(f'Full analysis set:       {len(X_all):,} events')

# ── Impute with median (fit on train only) ────────────────────────────────────
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_all_imp   = imputer.transform(X_all)

X_train_imputed = pd.DataFrame(X_train_imp, columns=FEATURE_COLS, index=X_train.index)
X_all_imputed   = pd.DataFrame(X_all_imp,   columns=FEATURE_COLS, index=X_all.index)

print(f'\nNaN after imputation (train): {X_train_imputed.isna().sum().sum()}')
print(f'NaN after imputation (all):   {X_all_imputed.isna().sum().sum()}')

# ── Compute class weights for balanced training ───────────────────────────────
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight('balanced', y_train)

# ── Train XGBoost ─────────────────────────────────────────────────────────────
model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=4,
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train_imputed, y_train, sample_weight=sample_weights, verbose=False)

# Quick validation
from sklearn.metrics import balanced_accuracy_score
y_pred_train = model.predict(X_train_imputed)
bal_acc = balanced_accuracy_score(y_train, y_pred_train)
print(f'\nTrain balanced accuracy: {bal_acc:.3f}')
print('Model trained successfully.')

Loaded 6,805 events x 104 columns
Year range: 1948 – 2008
Target distribution:
target
0    1301
1     273
2    3534
3    1697
Name: count, dtype: Int64

Using 45/45 candidate features

Train set (year < 1996): 4,286 events
Full analysis set:       6,805 events

NaN after imputation (train): 0
NaN after imputation (all):   0



Train balanced accuracy: 0.978
Model trained successfully.


## Cell 3: Compute SHAP values

In [3]:
# ── TreeExplainer on the full dataset ────────────────────────────────────────
explainer = shap.TreeExplainer(model)

# shap_values shape: (n_samples, n_features, n_classes) for multi-class XGBoost
shap_values = explainer(X_all_imputed)

print('SHAP values computed.')
print(f'shap_values.values shape: {shap_values.values.shape}')
# (6805, 45, 4)  -> samples x features x classes

# Convenience arrays
sv = shap_values.values          # (n, f, c)
n_samples, n_features, n_classes = sv.shape

# Mean |SHAP| across all classes: (n, f)
sv_mean_abs_classes = np.mean(np.abs(sv), axis=2)   # (n, f)

# Global importance: mean across samples
global_importance = sv_mean_abs_classes.mean(axis=0)  # (f,)
importance_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'mean_abs_shap': global_importance
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print('\nTop 10 features by mean |SHAP|:')
print(importance_df.head(10).to_string(index=False))

SHAP values computed.
shap_values.values shape: (6805, 45, 4)

Top 10 features by mean |SHAP|:
             feature  mean_abs_shap
 NUMBER_OF_Countries       0.372760
    events_prior_5yr       0.319435
cooperation_momentum       0.318031
                year       0.271787
         Issue_Type1       0.203924
    event_escalation       0.179861
    SP.POP.TOTL_wdi1       0.120820
     treaty_rate_5yr       0.110108
    SP.POP.TOTL_wdi2       0.101951
 ER.H2O.INTR.PC_wdi2       0.101739


## Cell 4: Global feature importance — mean |SHAP| bar chart

In [4]:
top20 = importance_df.head(20).copy()

# Readable feature labels
LABEL_MAP = {
    'cooperation_momentum':        'Cooperation momentum',
    'treaties_before_event':       'Treaties before event',
    'events_prior_5yr':            'Events (prior 5yr)',
    'treaty_rate_5yr':             'Treaty rate (5yr)',
    'event_escalation':            'Event escalation',
    'cold_war':                    'Cold War era',
    'year':                        'Year',
    'Issue_Type1':                 'Issue type',
    'bilateral':                   'Bilateral',
    'NUMBER_OF_Countries':         'No. countries',
    'NUMBER_OF_BASINS':            'No. basins',
    'NY.GDP.PCAP.CD_wdi1':         'GDP per capita (c1)',
    'NY.GDP.PCAP.CD_wdi2':         'GDP per capita (c2)',
    'SP.POP.TOTL_wdi1':            'Population (c1)',
    'SP.POP.TOTL_wdi2':            'Population (c2)',
    'MS.MIL.XPND.GD.ZS_wdi1':     'Military exp. % GDP (c1)',
    'MS.MIL.XPND.GD.ZS_wdi2':     'Military exp. % GDP (c2)',
    'ER.H2O.FWTL.ZS_wdi1':        'Freshwater withdrawal % (c1)',
    'ER.H2O.FWTL.ZS_wdi2':        'Freshwater withdrawal % (c2)',
    'ER.H2O.INTR.PC_wdi1':        'Internal freshwater (c1)',
    'ER.H2O.INTR.PC_wdi2':        'Internal freshwater (c2)',
    'InstitVuln_1':                'Institutional vulnerability (b1)',
    'InstitVuln_2':                'Institutional vulnerability (b2)',
    'HydroPolTe_1':                'Hydro-political tension (b1)',
    'HydroPolTe_2':                'Hydro-political tension (b2)',
    'NumberRipa_1':                'Riparian count (b1)',
    'NumberRipa_2':                'Riparian count (b2)',
    'Area_km2_1':                  'Basin area km² (b1)',
    'Area_km2_2':                  'Basin area km² (b2)',
    'Dams_Exist_1':                'Existing dams (b1)',
    'Dams_Exist_2':                'Existing dams (b2)',
    'Dam_Plnd_1':                  'Planned dams (b1)',
    'Dam_Plnd_2':                  'Planned dams (b2)',
    'EstDam24_1':                  'Dam storage (b1)',
    'EstDam24_2':                  'Dam storage (b2)',
    'runoff_1':                    'Runoff (b1)',
    'runoff_2':                    'Runoff (b2)',
    'withdrawal_1':                'Withdrawal (b1)',
    'withdrawal_2':                'Withdrawal (b2)',
    'consumpt_1':                  'Consumption (b1)',
    'consumpt_2':                  'Consumption (b2)',
    'Wetlands_k_1':                'Wetlands km² (b1)',
    'Wetlands_k_2':                'Wetlands km² (b2)',
    'PopDen2022_1':                'Population density (b1)',
    'PopDen2022_2':                'Population density (b2)',
}

top20['label'] = top20['feature'].map(LABEL_MAP).fillna(top20['feature'])

# Category colouring
def feature_category(f):
    if f in ['cooperation_momentum', 'treaties_before_event', 'events_prior_5yr',
              'treaty_rate_5yr', 'event_escalation', 'cold_war', 'year']:
        return 'Temporal / Treaty'
    elif 'wdi' in f:
        return 'Economic (WDI)'
    elif f in ['Issue_Type1', 'bilateral', 'NUMBER_OF_Countries', 'NUMBER_OF_BASINS']:
        return 'Event'
    else:
        return 'Hydrological / Institutional'

CAT_COLOURS = {
    'Temporal / Treaty':           '#1f77b4',
    'Economic (WDI)':              '#ff7f0e',
    'Event':                       '#9467bd',
    'Hydrological / Institutional':'#2ca02c',
}
top20['category'] = top20['feature'].apply(feature_category)
top20['colour']   = top20['category'].map(CAT_COLOURS)

fig_imp = go.Figure()
for cat, colour in CAT_COLOURS.items():
    sub = top20[top20['category'] == cat]
    if sub.empty:
        continue
    fig_imp.add_trace(go.Bar(
        x=sub['mean_abs_shap'],
        y=sub['label'],
        orientation='h',
        name=cat,
        marker_color=colour,
        marker_line_color='black',
        marker_line_width=0.5,
    ))

fig_imp.update_layout(
    template=pub_template,
    title=dict(text='Global Feature Importance (mean |SHAP|, all classes)', font_size=13),
    xaxis_title='Mean |SHAP value| (impact on model output)',
    yaxis_title='',
    yaxis=dict(categoryorder='total ascending'),
    barmode='stack',
    legend=dict(title='Feature category', x=0.55, y=0.05,
                bordercolor='black', borderwidth=1),
    height=560,
    width=700,
    font=dict(family='Arial', size=12, color='black'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgray', linecolor='black',
               linewidth=1, mirror=True, ticks='outside'),
    yaxis_linecolor='black',
    margin=dict(l=200, r=20, t=50, b=60),
)

fig_imp.show()
fig_imp.write_image(f'{FIGURES_DIR}/fig03a_shap_importance.png', scale=2)
fig_imp.write_html(f'{FIGURES_DIR}/fig03a_shap_importance.html')
print('Saved fig03a_shap_importance')

Saved fig03a_shap_importance


## Cell 5: SHAP beeswarm plot (top 15 features)

In [5]:
# ── Focus on class 3 (strong cooperation) — most policy-relevant ──────────────
CLASS_IDX = 3   # strong cooperation

top15_features = importance_df['feature'].head(15).tolist()
top15_labels   = [LABEL_MAP.get(f, f) for f in top15_features]

# Build long-form dataframe for scatter
rng = np.random.default_rng(seed=42)
rows = []
for rank, (feat, label) in enumerate(zip(top15_features, top15_labels)):
    fidx = FEATURE_COLS.index(feat)
    shap_col  = sv[:, fidx, CLASS_IDX]
    feat_col  = X_all_imputed[feat].values

    # Normalise feature values to [0, 1] for colouring
    vmin, vmax = np.nanpercentile(feat_col, 1), np.nanpercentile(feat_col, 99)
    feat_norm  = np.clip((feat_col - vmin) / (vmax - vmin + 1e-12), 0, 1)

    # Jitter on y-axis to avoid overplotting — subsample for speed
    n_pts = min(800, len(shap_col))
    idx   = rng.choice(len(shap_col), size=n_pts, replace=False)
    jitter = rng.uniform(-0.3, 0.3, size=n_pts)

    for i, j in zip(idx, jitter):
        rows.append({
            'feature': feat,
            'label':   label,
            'rank':    rank,
            'shap':    float(shap_col[i]),
            'feat_val': float(feat_norm[i]),
            'y':       float(rank + j),
        })

bee_df = pd.DataFrame(rows)

fig_bee = go.Figure()
fig_bee.add_trace(go.Scatter(
    x=bee_df['shap'],
    y=bee_df['y'],
    mode='markers',
    marker=dict(
        size=3,
        color=bee_df['feat_val'],
        colorscale='RdBu_r',
        cmin=0, cmax=1,
        colorbar=dict(
            title='Feature value<br>(low → high)',
            thickness=12,
            len=0.6,
            tickvals=[0, 0.5, 1],
            ticktext=['Low', 'Mid', 'High'],
            outlinecolor='black',
            outlinewidth=1,
        ),
        opacity=0.7,
        line=dict(width=0),
    ),
    showlegend=False,
))

# Zero-line
fig_bee.add_vline(x=0, line_dash='dash', line_color='gray', line_width=1)

fig_bee.update_layout(
    template=pub_template,
    title=dict(
        text=f'SHAP Beeswarm — Class: {CLASS_LABELS[CLASS_IDX]} (top 15 features)',
        font_size=13
    ),
    xaxis_title='SHAP value (impact on P(strong cooperation))',
    yaxis=dict(
        tickmode='array',
        tickvals=list(range(len(top15_features))),
        ticktext=top15_labels,
        showgrid=True, gridcolor='lightgray',
        linecolor='black', linewidth=1, mirror=True, ticks='outside',
    ),
    xaxis=dict(
        showgrid=False, linecolor='black', linewidth=1, mirror=True, ticks='outside'
    ),
    height=520,
    width=700,
    font=dict(family='Arial', size=12, color='black'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=200, r=90, t=50, b=60),
)

fig_bee.show()
fig_bee.write_image(f'{FIGURES_DIR}/fig03b_shap_beeswarm.png', scale=2)
fig_bee.write_html(f'{FIGURES_DIR}/fig03b_shap_beeswarm.html')
print('Saved fig03b_shap_beeswarm')

Saved fig03b_shap_beeswarm


## Cell 6: SHAP dependence plots for top 3 features

In [6]:
top3_features = importance_df['feature'].head(3).tolist()

def find_interaction_feature(target_feat, feature_list, sv_all, class_idx=3, top_n=5):
    """Return the feature with the highest SHAP interaction effect.
    Proxy: Pearson correlation between SHAP values of target feature
    and raw feature values of candidate features (excluding self).
    """
    tidx   = feature_list.index(target_feat)
    shap_t = sv_all[:, tidx, class_idx]
    best_feat, best_r = None, -1
    for f in feature_list:
        if f == target_feat:
            continue
        fidx = feature_list.index(f)
        vals = X_all_imputed[f].values
        mask = np.isfinite(shap_t) & np.isfinite(vals)
        if mask.sum() < 50:
            continue
        r = abs(np.corrcoef(shap_t[mask], vals[mask])[0, 1])
        if r > best_r:
            best_r, best_feat = r, f
    return best_feat

fig_dep = make_subplots(
    rows=1, cols=3,
    subplot_titles=[LABEL_MAP.get(f, f) for f in top3_features],
    horizontal_spacing=0.12,
)

for col_idx, feat in enumerate(top3_features, start=1):
    fidx = FEATURE_COLS.index(feat)
    shap_col  = sv[:, fidx, CLASS_IDX]     # CLASS_IDX = 3 from cell 5
    feat_vals = X_all_imputed[feat].values

    inter_feat = find_interaction_feature(feat, FEATURE_COLS, sv)
    inter_vals = X_all_imputed[inter_feat].values if inter_feat else np.zeros(len(shap_col))
    v0, v1     = np.nanpercentile(inter_vals, 1), np.nanpercentile(inter_vals, 99)
    inter_norm = np.clip((inter_vals - v0) / (v1 - v0 + 1e-12), 0, 1)

    inter_label = LABEL_MAP.get(inter_feat, inter_feat) if inter_feat else 'N/A'

    # Subsample for render speed
    rng2 = np.random.default_rng(99 + col_idx)
    idx2 = rng2.choice(len(shap_col), size=min(1500, len(shap_col)), replace=False)

    scatter = go.Scatter(
        x=feat_vals[idx2],
        y=shap_col[idx2],
        mode='markers',
        marker=dict(
            size=4,
            color=inter_norm[idx2],
            colorscale='Viridis',
            cmin=0, cmax=1,
            colorbar=dict(
                title=dict(text=inter_label, font_size=9),
                thickness=10,
                len=0.5,
                x=0.34 * col_idx - 0.05,
                tickvals=[0, 1],
                ticktext=['Low', 'High'],
                outlinewidth=1,
                outlinecolor='black',
            ) if col_idx == 3 else None,
            showscale=(col_idx == 3),
            opacity=0.65,
            line=dict(width=0),
        ),
        showlegend=False,
        name=feat,
    )
    fig_dep.add_trace(scatter, row=1, col=col_idx)
    fig_dep.add_hline(y=0, line_dash='dash', line_color='gray',
                      line_width=1, row=1, col=col_idx)

    # Axis labels
    fig_dep.update_xaxes(title_text=LABEL_MAP.get(feat, feat),
                         showgrid=False, linecolor='black', linewidth=1,
                         mirror=True, ticks='outside', row=1, col=col_idx)
    fig_dep.update_yaxes(title_text='SHAP value' if col_idx == 1 else '',
                         showgrid=True, gridcolor='lightgray',
                         linecolor='black', linewidth=1,
                         mirror=True, ticks='outside', row=1, col=col_idx)

fig_dep.update_layout(
    title=dict(
        text='SHAP Dependence Plots — Top 3 Features (P(strong cooperation))',
        font_size=13
    ),
    height=360,
    width=900,
    font=dict(family='Arial', size=11, color='black'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=60, r=80, t=60, b=60),
)

fig_dep.show()
fig_dep.write_image(f'{FIGURES_DIR}/fig03c_shap_dependence_top3.png', scale=2)
fig_dep.write_html(f'{FIGURES_DIR}/fig03c_shap_dependence_top3.html')
print('Saved fig03c_shap_dependence_top3')

Saved fig03c_shap_dependence_top3


## Cell 7: Temporal SHAP decomposition (novel contribution)

How do driver importances shift across geopolitical eras?  
**Hypothesis (Wolf 2007)**: institutional and treaty variables should gain predictive power post-Cold War as multilateral water governance matures.

In [7]:
# ── Define eras ───────────────────────────────────────────────────────────────
year_all_np = df_all['year'].fillna(-1).to_numpy().astype(int)
era_masks = {
    'Cold War (1948-1990)':      year_all_np < 1991,
    'Post-Cold War (1991-1999)': (year_all_np >= 1991) & (year_all_np < 2000),
    'Post-2000 (2000-2008)':     year_all_np >= 2000,
}

for era, mask in era_masks.items():
    print(f'{era}: {mask.sum():,} events')

# ── Mean |SHAP| per feature per era (averaged over all classes) ───────────────
# sv_mean_abs_classes: (n, f) — already averaged across classes
era_importance = {}
for era, mask in era_masks.items():
    era_imp = sv_mean_abs_classes[mask, :].mean(axis=0)
    era_importance[era] = dict(zip(FEATURE_COLS, era_imp))

# ── Top 10 features by overall importance ────────────────────────────────────
top10_features = importance_df['feature'].head(10).tolist()
top10_labels   = [LABEL_MAP.get(f, f) for f in top10_features]

# ── Grouped bar chart ─────────────────────────────────────────────────────────
fig_temp = go.Figure()

for era, colour in ERA_COLOURS.items():
    imp_values = [era_importance[era].get(f, 0) for f in top10_features]
    fig_temp.add_trace(go.Bar(
        name=era,
        x=top10_labels,
        y=imp_values,
        marker_color=colour,
        marker_line_color='black',
        marker_line_width=0.6,
        opacity=0.88,
    ))

fig_temp.update_layout(
    template=pub_template,
    title=dict(
        text='Temporal SHAP Decomposition: Feature Importance Across Geopolitical Eras',
        font_size=13
    ),
    barmode='group',
    xaxis=dict(
        tickangle=-35,
        showgrid=False, linecolor='black', linewidth=1, mirror=True, ticks='outside',
    ),
    yaxis=dict(
        title='Mean |SHAP value| (all classes)',
        showgrid=True, gridcolor='lightgray',
        linecolor='black', linewidth=1, mirror=True, ticks='outside',
    ),
    legend=dict(
        title='Era', x=0.63, y=0.98,
        bordercolor='black', borderwidth=1,
    ),
    height=450,
    width=850,
    font=dict(family='Arial', size=11, color='black'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=70, r=20, t=55, b=130),
)

fig_temp.show()
fig_temp.write_image(f'{FIGURES_DIR}/fig03d_temporal_shap.png', scale=2)
fig_temp.write_html(f'{FIGURES_DIR}/fig03d_temporal_shap.html')
print('Saved fig03d_temporal_shap')

# ── Print era-wise changes for key institutional features ─────────────────────
wolf_features = ['treaties_before_event', 'treaty_rate_5yr', 'InstitVuln_1',
                 'InstitVuln_2', 'cooperation_momentum']
print('\n── Institutional / treaty feature importance across eras ──────────────────')
era_list = list(ERA_COLOURS.keys())
header = f'{"Feature":<35}' + ''.join(f'{e[:20]:<22}' for e in era_list)
print(header)
print('-' * len(header))
for feat in wolf_features:
    if feat not in FEATURE_COLS:
        continue
    row = f'{LABEL_MAP.get(feat, feat):<35}'
    for era in era_list:
        val = era_importance[era].get(feat, 0)
        row += f'{val:<22.4f}'
    print(row)

Cold War (1948-1990): 2,721 events
Post-Cold War (1991-1999): 2,561 events
Post-2000 (2000-2008): 1,523 events


Saved fig03d_temporal_shap

── Institutional / treaty feature importance across eras ──────────────────
Feature                            Cold War (1948-1990)  Post-Cold War (1991-  Post-2000 (2000-2008  
-----------------------------------------------------------------------------------------------------
Treaties before event              0.1057                0.0931                0.0851                
Treaty rate (5yr)                  0.0947                0.1039                0.1482                
Institutional vulnerability (b1)   0.0191                0.0184                0.0149                
Institutional vulnerability (b2)   0.0056                0.0067                0.0071                
Cooperation momentum               0.3486                0.3214                0.2578                


## Cell 8: Case study SHAP force plots (waterfall-style)

Four canonical transboundary river basins: Indus (India-Pakistan), Nile, Jordan, Mekong.

In [8]:
def find_case_event(df_src, bccode_pattern, bar_condition, label):
    """Return the index of the best matching event in df_all."""
    mask = df_src['BCCODE1'].str.contains(bccode_pattern, na=False, regex=True)
    sub  = df_src[mask]
    if sub.empty:
        print(f'  [{label}] No events found for pattern "{bccode_pattern}"')
        return None
    if bar_condition == 'conflict':
        sub2 = sub[sub['BAR_Scale'] < 0]
    else:
        sub2 = sub[sub['BAR_Scale'] > 3]
    if sub2.empty:
        print(f'  [{label}] No {bar_condition} event found; using extreme available')
        sub2 = sub.sort_values('BAR_Scale') if bar_condition == 'conflict' else sub.sort_values('BAR_Scale', ascending=False)
    chosen = sub2.iloc[0]
    print(f'  [{label}] {bar_condition}: idx={chosen.name}, '
          f'year={int(chosen["year"])}, BAR={chosen["BAR_Scale"]:.1f}, '
          f'target={chosen["target"]}, BCCODE1={chosen["BCCODE1"]}')
    return chosen.name

CASES = [
    ('INDU',         'conflict',    'Indus conflict'),
    ('INDU',         'cooperation', 'Indus cooperation'),
    ('NILE',         'conflict',    'Nile conflict'),
    ('NILE',         'cooperation', 'Nile cooperation'),
    ('JORD',         'conflict',    'Jordan conflict'),
    ('JORD',         'cooperation', 'Jordan cooperation'),
    ('MEKO',         'conflict',    'Mekong conflict'),
    ('MEKO',         'cooperation', 'Mekong cooperation'),
]

print('Selecting case study events:')
case_indices = []
for pattern, cond, label in CASES:
    idx = find_case_event(df_all, pattern, cond, label)
    case_indices.append((idx, label))

# ── Drop cases with no valid index ────────────────────────────────────────────
valid_cases = [(idx, lbl) for idx, lbl in case_indices if idx is not None]
print(f'\n{len(valid_cases)} valid case events found.')

Selecting case study events:
  [Indus conflict] conflict: idx=1, year=1948, BAR=-3.0, target=0, BCCODE1=INDU_IND
  [Indus cooperation] cooperation: idx=235, year=1954, BAR=4.0, target=3, BCCODE1=INDU_PAK
  [Nile conflict] conflict: idx=284, year=1954, BAR=-2.0, target=0, BCCODE1=NILE_EGY
  [Nile cooperation] cooperation: idx=46, year=1949, BAR=4.0, target=3, BCCODE1=NILE_EGY
  [Jordan conflict] conflict: idx=42, year=1948, BAR=-6.0, target=0, BCCODE1=JORD_EGY
  [Jordan cooperation] cooperation: idx=165, year=1953, BAR=4.0, target=3, BCCODE1=JORD_JOR
  [Mekong conflict] conflict: idx=2982, year=1992, BAR=-2.0, target=0, BCCODE1=MEKO_THA
  [Mekong cooperation] cooperation: idx=489, year=1957, BAR=4.0, target=3, BCCODE1=MEKO_THA

8 valid case events found.


In [9]:
def waterfall_plotly(event_idx, label, sv_array, feature_names, feature_labels,
                     X_imp, class_idx=3, top_n=10):
    """
    Return a Plotly waterfall figure for a single event,
    showing the top_n SHAP contributions for the specified class.
    """
    # Locate position in X_all_imputed (same index order)
    pos = X_imp.index.get_loc(event_idx)
    shap_event = sv_array[pos, :, class_idx]   # (n_features,)
    feat_vals  = X_imp.iloc[pos]

    # Base value (expected value for this class)
    base_val = explainer.expected_value[class_idx] if isinstance(
        explainer.expected_value, (list, np.ndarray)) else explainer.expected_value

    # Select top_n by absolute SHAP
    order   = np.argsort(np.abs(shap_event))[::-1][:top_n]
    rest_sv = shap_event[np.argsort(np.abs(shap_event))[::-1][top_n:]].sum()

    feats_sel  = [feature_names[i]  for i in order]
    labels_sel = [feature_labels[i] for i in order]
    shap_sel   = shap_event[order]
    vals_sel   = [feat_vals[f] for f in feats_sel]

    # Append "other" bar
    labels_sel = np.append(labels_sel, 'All other features')
    shap_sel   = np.append(shap_sel, rest_sv)

    # Reverse order so most important is on top
    labels_sel = labels_sel[::-1]
    shap_sel   = shap_sel[::-1]

    colours = ['#d62728' if v < 0 else '#1f77b4' for v in shap_sel]

    fig = go.Figure(go.Bar(
        x=shap_sel,
        y=labels_sel,
        orientation='h',
        marker_color=colours,
        marker_line_color='black',
        marker_line_width=0.5,
    ))
    fig.add_vline(x=0, line_dash='dash', line_color='gray', line_width=1)

    fig.update_layout(
        title=dict(text=label, font_size=11),
        xaxis=dict(
            title='SHAP value (P(strong coop.))',
            showgrid=False, linecolor='black', linewidth=1,
            mirror=True, ticks='outside', title_font_size=10,
        ),
        yaxis=dict(
            showgrid=True, gridcolor='lightgray',
            linecolor='black', linewidth=1,
            mirror=True, ticks='outside',
            tickfont_size=9,
        ),
        font=dict(family='Arial', size=10, color='black'),
        plot_bgcolor='white',
        paper_bgcolor='white',
        height=340,
        width=440,
        margin=dict(l=165, r=20, t=40, b=50),
    )
    return fig


feature_label_list = [LABEL_MAP.get(f, f) for f in FEATURE_COLS]

# ── Build a composite figure (2 columns, n_cases/2 rows) ─────────────────────
n_valid = len(valid_cases)
ncols   = 2
nrows   = int(np.ceil(n_valid / ncols))

fig_cs = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[lbl for _, lbl in valid_cases],
    horizontal_spacing=0.25,
    vertical_spacing=0.12,
)

for k, (event_idx, label) in enumerate(valid_cases):
    row = k // ncols + 1
    col = k %  ncols + 1
    pos = X_all_imputed.index.get_loc(event_idx)
    shap_event = sv[pos, :, CLASS_IDX]
    base_val   = (explainer.expected_value[CLASS_IDX]
                  if isinstance(explainer.expected_value, (list, np.ndarray))
                  else explainer.expected_value)

    order   = np.argsort(np.abs(shap_event))[::-1][:10]
    rest_sv = shap_event[np.argsort(np.abs(shap_event))[::-1][10:]].sum()
    labels_sub = [LABEL_MAP.get(FEATURE_COLS[i], FEATURE_COLS[i]) for i in order]
    shap_sub   = list(shap_event[order])
    labels_sub.append('All other features')
    shap_sub.append(rest_sv)
    labels_sub = labels_sub[::-1]
    shap_sub   = shap_sub[::-1]
    colours    = ['#d62728' if v < 0 else '#1f77b4' for v in shap_sub]

    fig_cs.add_trace(
        go.Bar(
            x=shap_sub,
            y=labels_sub,
            orientation='h',
            marker_color=colours,
            marker_line_color='black',
            marker_line_width=0.4,
            showlegend=False,
            name=label,
        ),
        row=row, col=col
    )
    fig_cs.add_vline(x=0, line_dash='dash', line_color='gray',
                     line_width=1, row=row, col=col)
    fig_cs.update_xaxes(
        showgrid=False, linecolor='black', linewidth=1,
        mirror=True, ticks='outside',
        title_text='SHAP (P(strong coop.))' if row == nrows else '',
        row=row, col=col
    )
    fig_cs.update_yaxes(
        showgrid=True, gridcolor='lightgray',
        linecolor='black', linewidth=1,
        mirror=True, ticks='outside', tickfont_size=8,
        row=row, col=col
    )

fig_cs.update_layout(
    title=dict(
        text='Case Study SHAP Waterfall Plots — Top 10 Feature Contributions per Event',
        font_size=13
    ),
    height=nrows * 340,
    width=1000,
    font=dict(family='Arial', size=10, color='black'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=165, r=20, t=60, b=60),
)

fig_cs.show()
fig_cs.write_image(f'{FIGURES_DIR}/fig03e_case_studies.png', scale=2)
fig_cs.write_html(f'{FIGURES_DIR}/fig03e_case_studies.html')
print('Saved fig03e_case_studies')

Saved fig03e_case_studies


## Cell 9: Summary statistics and key findings

In [10]:
# ── 1. Global top-5 features ──────────────────────────────────────────────────
print('=' * 70)
print('OVERALL TOP 5 FEATURES BY MEAN |SHAP|')
print('=' * 70)
for rank, row in importance_df.head(5).iterrows():
    lbl = LABEL_MAP.get(row['feature'], row['feature'])
    print(f'  {rank+1}. {lbl:<40} {row["mean_abs_shap"]:.4f}')

# ── 2. Era transitions for each top-10 feature ───────────────────────────────
print()
print('=' * 70)
print('FEATURE IMPORTANCE SHIFT ACROSS GEOPOLITICAL ERAS (top-10, all classes)')
print('=' * 70)
era_list = list(ERA_COLOURS.keys())
col_w = 24
header = f'{"Feature":<36}' + ''.join(f'{e[:col_w]:<{col_w}}' for e in era_list) + '  Trend'
print(header)
print('-' * len(header))
for feat in importance_df['feature'].head(10).tolist():
    lbl = LABEL_MAP.get(feat, feat)
    vals = [era_importance[e].get(feat, 0) for e in era_list]
    # Simple trend: compare Cold War vs Post-2000
    delta = vals[-1] - vals[0]
    trend = 'RISING  (+)' if delta > 0.005 else ('FALLING (-)' if delta < -0.005 else 'STABLE  (~)')
    row_str = f'{lbl:<36}' + ''.join(f'{v:<{col_w}.4f}' for v in vals) + f'  {trend}'
    print(row_str)

# ── 3. Wolf hypothesis test ───────────────────────────────────────────────────
print()
print('=' * 70)
print("WOLF'S HYPOTHESIS: Institutional features become more predictive over time?")
print('=' * 70)
wolf_check = {
    'treaties_before_event': 'Treaties before event',
    'treaty_rate_5yr':       'Treaty rate (5yr)',
    'InstitVuln_1':          'Institutional vulnerability (b1)',
    'InstitVuln_2':          'Institutional vulnerability (b2)',
    'cooperation_momentum':  'Cooperation momentum',
    'cold_war':              'Cold War indicator',
}
for feat, lbl in wolf_check.items():
    if feat not in FEATURE_COLS:
        continue
    cw  = era_importance['Cold War (1948-1990)'].get(feat, 0)
    pcw = era_importance['Post-Cold War (1991-1999)'].get(feat, 0)
    p2k = era_importance['Post-2000 (2000-2008)'].get(feat, 0)
    delta_pct = (p2k - cw) / (cw + 1e-9) * 100
    direction = 'SUPPORTS Wolf' if p2k > cw else 'AGAINST Wolf'
    print(f'  {lbl:<38}  CW={cw:.4f}  PostCW={pcw:.4f}  Post2k={p2k:.4f}'
          f'  Δ={delta_pct:+.1f}%  [{direction}]')

# ── 4. Expected value reference ───────────────────────────────────────────────
print()
print('=' * 70)
print('EXPLAINER BASE VALUES (expected model output per class)')
print('=' * 70)
ev = explainer.expected_value
if hasattr(ev, '__len__'):
    for ci, v in enumerate(ev):
        print(f'  Class {ci} ({CLASS_LABELS[ci]}): {v:.4f}')
else:
    print(f'  Base value: {ev:.4f}')

print()
print('All SHAP figures saved to:', os.path.abspath(FIGURES_DIR))

OVERALL TOP 5 FEATURES BY MEAN |SHAP|
  1. No. countries                            0.3728
  2. Events (prior 5yr)                       0.3194
  3. Cooperation momentum                     0.3180
  4. Year                                     0.2718
  5. Issue type                               0.2039

FEATURE IMPORTANCE SHIFT ACROSS GEOPOLITICAL ERAS (top-10, all classes)
Feature                             Cold War (1948-1990)    Post-Cold War (1991-1999Post-2000 (2000-2008)     Trend
-------------------------------------------------------------------------------------------------------------------
No. countries                       0.2867                  0.4928                  0.3247                    RISING  (+)
Events (prior 5yr)                  0.2580                  0.3591                  0.3624                    RISING  (+)
Cooperation momentum                0.3486                  0.3214                  0.2578                    FALLING (-)
Year                      